# ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) Project Capstone

## Forecasting and Optimizing Solar Power: Predictive and Reinforcement Learning Approaches

---

[README](../README.md) | **Part 1: EDA** | [Part 2: xx](02_Vectorizer.ipynb)

---

### Introduction

https://www.kaggle.com/datasets/stefancomanita/hourly-electricity-consumption-and-production

https://www.visualcrossing.com/weather/weather-data-services/Bucharest,Romania/metric/

https://thingler.io/country/Romania


<!-- def create_hourly_csv(filename):
    # Define start and end datetime
    start = "2024-01-01 00:00"
    end = "2024-03-31 23:00"
    
    # Generate date range with hourly frequency
    date_range = pd.date_range(start=start, end=end, freq='H')
    
    # Create a DataFrame
    df = pd.DataFrame({"dt": date_range})
    
    # Save to CSV
    df.to_csv(filename, index=False)
    print(f"CSV file '{filename}' created successfully!")

# Call the function
create_hourly_csv("hourly_data.csv") -->

### Import & Cleaning

#### Essential Libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder

#### Weather Data
- Weather data
  - The data was extracted from [Visual Crossing's weather data services](https://www.visualcrossing.com/weather/weather-data-services/Bucharest,Romania/metric/), covering a period of three months.
  - The extracted data is saved into three files, [bucharest_weather_jan24.csv]('../data/bucharest_weather_jan24.csv'), [bucharest_weather_feb24.csv]('../data/bucharest_weather_feb24.csv'), and [bucharest_weather_mar24.csv]('../data/bucharest_weather_mar24.csv').

In [11]:
# Import weather data from three files as DataFrames
weather_jan = pd.read_csv('../data/bucharest_weather_jan24.csv')
weather_feb = pd.read_csv('../data/bucharest_weather_feb24.csv')
weather_mar = pd.read_csv('../data/bucharest_weather_mar24.csv')

In [12]:
# Check data shapes
# These DataFrames share the same number of columns; however, weather_mar has 1 missing row (it should have 744 rows)
weather_jan.shape, weather_feb.shape, weather_mar.shape

((744, 24), (696, 24), (743, 24))

In [13]:
# Concatenate those DataFrames
weather = pd.concat([weather_jan, weather_feb, weather_mar]
                    , axis = 0
                    , ignore_index = True
                   )

In [14]:
weather.head(1)

,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,sealevelpressure,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
0,"Bucharest,Romania",2024-01-01T00:00:00,3.4,3.4,2.9,96.47,0.0,0,NaN,0.0,...,1018.1,0.0,4.4,0.0,0.0,0.0,NaN,Clear,clear-night,"F0727,15422099999,15419099999,15455099999,1542..."


In [15]:
# Check the number of zeros, NaNs, number of unique values
percent_zeros = (weather == 0).mean() * 100
percent_null = weather.isnull().mean() * 100
unique_values = weather.nunique()

# Create a DataFrame with data type, % zeros, % missing values, and number of unique values
column_info = pd.DataFrame({
    'Data Type': weather.dtypes
    , '% Zeros': percent_zeros
    , '% Missing Values': percent_null
    , 'Unique Values': unique_values
})

column_info

,Data Type,% Zeros,% Missing Values,Unique Values
name,object,0.000000,0.000000,2
datetime,object,0.000000,0.000000,2183
temp,float64,0.229043,0.000000,300
feelslike,float64,0.320660,0.000000,318
dew,float64,0.549702,0.000000,218
humidity,float64,0.000000,0.000000,1816
precip,float64,96.930829,0.000000,57
precipprob,int64,96.930829,0.000000,2
preciptype,object,0.000000,96.472744,3
snow,float64,99.450298,0.000000,10


In [16]:
# Drop columns with too many zeros, missing values, too few unique values, or meaningless data

weather.drop(columns = ['name'                 # meaningless data: Bucharest,Romania; bucharest 
                        , 'precip'             # too many zeros
                        , 'precipprob'         # too many zeros
                        , 'preciptype'         # too many missing values
                        , 'snow'               # too many zeros
                        , 'solarenergy'        # too many zeros
                        , 'uvindex'            # Only zeros and NaNs
                        , 'severerisk'         # All Zeros
                        , 'conditions'         # meaningless
                        , 'stations'           # meaningless
                       ]
             , errors = 'ignore'
             , inplace = True)

In [17]:
# Convert to binary and drop the original columns
weather['has_snow'] = (weather['snowdepth'] > 0).astype(int)
weather['has_solarradiation'] = (weather['solarradiation'] > 0).astype(int)

weather.drop(columns = ['snowdepth'
                        , 'solarradiation'
                       ]
             , errors = 'ignore'
             , inplace = True)

In [18]:
# Map 'partly-cloudy-day' and 'partly-cloudy-night' to 'partly-cloudy'
# Map 'clear-night' and 'clear-day' to 'clear'

weather['icon'] = weather['icon'].replace({
    'partly-cloudy-night': 'partly-cloudy',
    'partly-cloudy-day': 'partly-cloudy',
    'clear-night': 'clear',
    'clear-day': 'clear'
})

# Check the value counts
weather['icon'].value_counts(dropna = False)

icon
cloudy           876
partly-cloudy    791
clear            437
rain              52
snow              15
fog               12
Name: count, dtype: int64

In [19]:
# Initialize OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)

# Fit and transform the 'icon' column
encoded_icons = encoder.fit_transform(weather[['icon']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded_icons, columns=encoder.get_feature_names_out(['icon']))

# Concatenate the original weather DataFrame with the encoded columns, and drop the original 'icon' and 'icon_clear' columns
weather = pd.concat([weather, encoded_df], axis=1).drop(['icon', 'icon_clear'], axis=1)

# Check the result
weather.head()

,datetime,temp,feelslike,dew,humidity,windgust,windspeed,winddir,sealevelpressure,cloudcover,visibility,has_snow,has_solarradiation,icon_cloudy,icon_fog,icon_partly-cloudy,icon_rain,icon_snow
0,2024-01-01T00:00:00,3.4,3.4,2.9,96.47,6.1,3.7,19.0,1018.1,0.0,4.4,0,0,0.0,0.0,0.0,0.0,0.0
1,2024-01-01T01:00:00,3.1,3.1,2.6,96.54,6.5,3.7,18.0,1017.7,0.0,1.6,0,0,0.0,0.0,0.0,0.0,0.0
2,2024-01-01T02:00:00,2.1,2.1,1.8,97.83,6.1,3.4,10.0,1017.5,0.0,1.6,0,0,0.0,0.0,0.0,0.0,0.0
3,2024-01-01T03:00:00,1.6,1.6,1.4,98.49,6.5,0.3,354.0,1017.2,0.0,1.6,0,0,0.0,0.0,0.0,0.0,0.0
4,2024-01-01T04:00:00,1.5,1.5,1.2,97.81,6.8,0.3,315.0,1016.9,0.0,1.6,0,0,0.0,0.0,0.0,0.0,0.0


In [30]:
# Formatting datetime
weather['datetime'] = pd.to_datetime(weather['datetime'])

In [36]:
# Find dates with incomplete data
# weather.groupby(weather['datetime'].dt.date).size().loc[lambda x: x != 24]

# Output:
# 2024-03-31    23
# dtype: int64

# The date with the missing row is '2024-03-31'

In [42]:
# Find the missing row for '2024-03-31'
# weather.loc[weather['datetime'].dt.date == pd.Timestamp('2024-03-01').date(), 'datetime']

# Output:
# 1440   2024-03-01 00:00:00
# 1441   2024-03-01 01:00:00
# 1442   2024-03-01 02:00:00
# 1443   2024-03-01 03:00:00
# 1444   2024-03-01 04:00:00
# 1445   2024-03-01 05:00:00
# 1446   2024-03-01 06:00:00
# 1447   2024-03-01 07:00:00
# 1448   2024-03-01 08:00:00
# 1449   2024-03-01 09:00:00
# 1450   2024-03-01 10:00:00
# 1451   2024-03-01 11:00:00
# 1452   2024-03-01 12:00:00
# 1453   2024-03-01 13:00:00
# 1454   2024-03-01 14:00:00
# 1455   2024-03-01 15:00:00
# 1456   2024-03-01 16:00:00
# 1457   2024-03-01 17:00:00
# 1458   2024-03-01 18:00:00
# 1459   2024-03-01 19:00:00
# 1460   2024-03-01 20:00:00
# 1461   2024-03-01 21:00:00
# 1462   2024-03-01 22:00:00
# 1463   2024-03-01 23:00:00
# Name: datetime, dtype: datetime64[ns]

# The missing row is '2024-03-31 03:00'

In [44]:
# Append the missing row and sort the index
weather = pd.concat([weather, pd.DataFrame({'datetime': [pd.Timestamp('2024-03-31 03:00')]})])